# Stage 3: Cooldown (батч ×4, финальный дотрейс)

Зеркало фазы охлаждения TinyLlama v1.1: в финале обучения размер батча
увеличивается (1.8M → 7.2M токенов, ×4), LR — на минимуме. Это помогает
найти стабильный локальный минимум и финализирует модель.

У нас: `gradient_accumulation_steps=4` при batch 32 × block 512 = 16384 →
65536 токенов на шаг (ровно ×4 от pretrain), LR 1e-5 → 1e-6, короткий прогон
из лучшего чекпойнта `2.continue_pretrain`. Тот же микс данных (cooldown = то
же распределение, меняется только размер батча).

Запуск: cwd = `3.cooldown`.

In [ ]:
# === Setup: среда + зависимости (локально / devcontainer / Colab) ===
import sys

IN_COLAB = "google.colab" in sys.modules
MLFLOW_ENABLED = not IN_COLAB  # в Colab локального MLflow-сервера нет

if IN_COLAB:
    import subprocess
    from pathlib import Path
    PROJECT_ROOT = Path("/content/spartan-torch")
    if not (PROJECT_ROOT / ".git").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/mievst/spartan-torch.git", str(PROJECT_ROOT)],
            check=True,
        )
    # репо-модули экспериментов (tinyllama/*.py, vit/vision_transformer.py) + исходники lib
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
    sys.path.insert(0, str(PROJECT_ROOT / "experiments"))
    # Colab приходит со своим numpy/scipy; наш -e ресолв мог рассогласовать их.
    # Апгрейдим пару вместе, чтобы они совпали (иначе datasets -> scipy падает
    # на numpy._core._multiarray_umath._blas_supports_fpe).
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "--upgrade-strategy", "eager", "numpy", "scipy"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e",
         f"{PROJECT_ROOT}[experiments,dev]"],
        check=True,
    )
else:
    PROJECT_ROOT = None

print(f"IN_COLAB={IN_COLAB} | MLFLOW_ENABLED={MLFLOW_ENABLED} | PROJECT_ROOT={PROJECT_ROOT} | python={sys.version.split()[0]}")


## Отклонения от статьи

См. Stage 1 (pretrain) — архитектура и данные идентичны.
Stage 3 отличается только LR, батчём и объёмом данных.

## 0. Настройки

In [ ]:
from pathlib import Path
import sys

import torch

ROOT = (PROJECT_ROOT / "experiments/llm/tinyllama/3.cooldown") if IN_COLAB else Path.cwd()
EXPERIMENT_ROOT = ROOT.parent
sys.path.insert(0, str(EXPERIMENT_ROOT))  # tinyllama/

DATA_DIR = EXPERIMENT_ROOT / "data"
CKPT_DIR = ROOT / "checkpoints"
PRETRAIN_DIR = EXPERIMENT_ROOT / "2.continue_pretrain" / "checkpoints" / "best"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"cwd: {ROOT} | device: {DEVICE} | ckpt: {PRETRAIN_DIR}")

SEED = 0
torch.manual_seed(SEED)

# --- данные: тот же микс, что в continue_pretrain ---
MIX = "cooldown"
BLOCK_SIZE = 512
TOTAL_BLOCKS = 125000

# --- тренер: батч ×4 (grad_accum), LR на минимуме, коротко ---
EPOCHS = 3
BATCH_SIZE = 32
GRAD_ACCUM = 4  # 32 * 4 * 512 = 65536 токенов/шаг (×4 от pretrain 16384)
LR = 1e-5
WEIGHT_DECAY = 0.1
WARMUP_STEPS = 50
MIN_LR = 1e-6
EVAL_STEPS = 250
SAVE_STEPS = 250

# --- MLflow ---
MLFLOW_TRACKING_URI = "http://host.docker.internal:5000"
MLFLOW_EXPERIMENT_NAME = "tinyllama-cooldown"

torch.set_float32_matmul_precision("medium")

assert PRETRAIN_DIR.exists(), f"continue checkpoint not found: {PRETRAIN_DIR}"

## 1. Данные: микс cooldown (кэши из `../data/`)

In [ ]:
from data import build_eval_dataset, load_tokenizer, make_mixed_dataset

tokenizer = load_tokenizer(DATA_DIR / "tokenizer")

train_ds = make_mixed_dataset(
    MIX,
    DATA_DIR / "blocks",
    tokenizer,
    BLOCK_SIZE,
    total_blocks=TOTAL_BLOCKS,
    seed=SEED,
)
val_ds = build_eval_dataset(tokenizer, BLOCK_SIZE, cache_dir=DATA_DIR / "blocks")
print(f"train blocks={len(train_ds):,} | eval blocks={len(val_ds):,}")

## 2. Модель из continue-pretrain чекпойнта

In [ ]:
from model import CausalLM

model = CausalLM.from_pretrained(PRETRAIN_DIR)
x = torch.randint(0, model.config.vocab_size, (2, model.config.block_size))
with torch.no_grad():
    loss = model(input_ids=x, labels=x).loss
print(f"ckpt: {PRETRAIN_DIR} | params: {model.num_params():,} | fwd ok | loss={loss.item():.3f}")

## 3. HF Trainer (батч ×4)

In [ ]:
from transformers import DataCollatorForLanguageModeling, EarlyStoppingCallback, Trainer, TrainingArguments

import mlflow
from train import SampleTextCallback, report_to_value

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

args = TrainingArguments(
    output_dir=str(CKPT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr": MIN_LR},
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=20,
    bf16=(DEVICE == "cuda"),
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    torch_compile=(DEVICE == "cuda"),
    seed=SEED,
    report_to=report_to_value(MLFLOW_TRACKING_URI),
    run_name=MLFLOW_EXPERIMENT_NAME,
    dataloader_pin_memory=False,
)

sample_cb = SampleTextCallback(
    model,
    tokenizer,
    prompts=["The history of Rome", "Machine learning is", "def fibonacci(n):"],
    max_new_tokens=64,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    callbacks=[sample_cb, EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()

## 4. Проверка

In [ ]:
from train import evaluate_ppl, generate

model = trainer.model
trainer.save_model(str(CKPT_DIR / "best"))
print("best ckpt:", CKPT_DIR / "best")

ppl = evaluate_ppl(model, val_ds, tokenizer, device=DEVICE)
print(f"val perplexity: {ppl:.2f}")

print("\nsample (text):")
print(generate(model, tokenizer, "The history of Rome", max_new_tokens=48))
print("\nsample (code):")
print(generate(model, tokenizer, "def fibonacci(n):", max_new_tokens=48))